In [ ]:
import os
import duckdb
import boto3
from dotenv import load_dotenv
import pandas as pd

Configuração do Pandas

In [ ]:
pd.set_option('display.max_columns', None)

Carregando as variáveis do arquivo .env

In [ ]:
load_dotenv()

PROFILE = os.getenv("AWS_PROFILE")
REGION = os.getenv("AWS_REGION")
PREFIX = os.getenv("PROJECT_BUCKET_PREFIX")
BRONZE_BUCKET = f"{PREFIX}-bronze"

Definindo as conexões com o S3 e o DuckDB

In [ ]:
session = boto3.Session(profile_name=PROFILE, region_name=REGION)
creds = session.get_credentials().get_frozen_credentials()

con = duckdb.connect(':memory:')
con.execute("INSTALL httpfs; LOAD httpfs; INSTALL aws; LOAD aws;")
con.execute(f"SET s3_region='{REGION}';")
con.execute(f"SET s3_access_key_id='{creds.access_key}';")
con.execute(f"SET s3_secret_access_key='{creds.secret_key}';")

Carregando arquivo do Bucket S3

In [ ]:
parquet_path = f"s3://{BRONZE_BUCKET}/perfil_comparecimento_abstencao/2022/perfil_comparecimento_abstencao_2022_BRASIL.parquet"

#### Schema do Arquivo

In [ ]:
con.execute(f"DESCRIBE SELECT * FROM '{parquet_path}'").df()


#### Amostra dos Dados

In [ ]:
con.execute(f"SELECT * FROM '{parquet_path}' LIMIT 10").df()